# 08 – Delta-Hedged PnL

Simulate a discrete delta hedge of a short call and plot the hedged PnL. Demonstrates how hedging frequency and vol realisation affect residual PnL.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib.pyplot as plt
from pricer.vanilla_bs import bs_price
from pricer.greeks_bs import bs_delta

np.random.seed(42)


## Parameters

In [ ]:
S0      = 100.0
K       = 100.0
T       = 1.0
r       = 0.05
sigma   = 0.20    # vol used to hedge (model vol)
sigma_r = 0.22    # vol actually realised in the simulation
q       = 0.0
n_steps = 252     # daily rehedging
dt      = T / n_steps


## Simulate a GBM path under realised vol

In [ ]:
drift_r    = (r - 0.5 * sigma_r ** 2) * dt
diffusion_r = sigma_r * np.sqrt(dt)
Z           = np.random.normal(size=n_steps)

S = np.zeros(n_steps + 1)
S[0] = S0
for i in range(n_steps):
    S[i + 1] = S[i] * np.exp(drift_r + diffusion_r * Z[i])

plt.figure(figsize=(9, 4))
plt.plot(S)
plt.title("Simulated asset path")
plt.xlabel("Day")
plt.ylabel("Price")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Discrete delta hedge

We are short one call. Each step we rebalance our stock holding to delta-neutralise.

In [ ]:
option_price_0 = bs_price(S0, K, T, r, sigma, "call", q)
print(f"Initial call price: {option_price_0:.4f}")

# Cash account starts with the premium received
cash   = option_price_0
delta_held = bs_delta(S0, K, T, r, sigma, "call", q)
cash  -= delta_held * S0   # buy initial delta hedge

pnl = np.zeros(n_steps + 1)
pnl[0] = 0.0

for i in range(1, n_steps + 1):
    t_remaining = T - i * dt
    cash *= np.exp(r * dt)  # accrue interest on cash

    if t_remaining > 1e-9:
        new_delta = bs_delta(S[i], K, t_remaining, r, sigma, "call", q)
    else:
        new_delta = 1.0 if S[i] > K else 0.0

    # rebalance stock position
    cash -= (new_delta - delta_held) * S[i]
    delta_held = new_delta

    # portfolio value = cash + delta*S - call_value
    if t_remaining > 1e-9:
        call_val = bs_price(S[i], K, t_remaining, r, sigma, "call", q)
    else:
        call_val = max(S[i] - K, 0.0)

    pnl[i] = cash + delta_held * S[i] - call_val

print(f"Final hedged PnL: {pnl[-1]:.4f}")


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(pnl)
plt.axhline(0, linestyle="--", color="grey", alpha=0.5)
plt.title(f"Delta-hedged PnL  (model vol={sigma:.0%}, realised vol={sigma_r:.0%})")
plt.xlabel("Day")
plt.ylabel("PnL ($)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## Multi-path distribution of terminal PnL

Run many paths to see the distribution of final hedge PnL.

In [ ]:
n_paths = 2000
terminal_pnl = []

for _ in range(n_paths):
    Z_path = np.random.normal(size=n_steps)
    S_path = np.zeros(n_steps + 1)
    S_path[0] = S0
    for i in range(n_steps):
        S_path[i + 1] = S_path[i] * np.exp(drift_r + diffusion_r * Z_path[i])

    cash_p = option_price_0
    dh = bs_delta(S0, K, T, r, sigma, "call", q)
    cash_p -= dh * S0

    for i in range(1, n_steps + 1):
        t_rem = T - i * dt
        cash_p *= np.exp(r * dt)
        new_dh = bs_delta(S_path[i], K, t_rem, r, sigma, "call", q) if t_rem > 1e-9 else (1.0 if S_path[i] > K else 0.0)
        cash_p -= (new_dh - dh) * S_path[i]
        dh = new_dh

    final_call = max(S_path[-1] - K, 0.0)
    terminal_pnl.append(cash_p + dh * S_path[-1] - final_call)

print(f"Mean PnL:   {np.mean(terminal_pnl):.4f}")
print(f"Std PnL:    {np.std(terminal_pnl):.4f}")
print(f"5th pct:    {np.percentile(terminal_pnl, 5):.4f}")
print(f"95th pct:   {np.percentile(terminal_pnl, 95):.4f}")

plt.figure(figsize=(9, 4))
plt.hist(terminal_pnl, bins=60, edgecolor="white")
plt.axvline(0, linestyle="--", color="red", label="zero")
plt.title(f"Terminal hedged PnL distribution  ({n_paths} paths)")
plt.xlabel("PnL ($)")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
